# 🎥 TeleConf-Enhance
**Real-time Video Enhancement Pipeline for Teleconferencing**

**Stage 1**: SAM2 (Meta AI, 2024) — Background Removal  
**Stage 2**: RIFE v4.25 (ECCV 2022) — Frame Interpolation (30fps → 60fps)

---
> Portfolio project targeting **Microsoft Applied Sciences Group** internship.
> JD keywords: *frame interpolation, AR for tele-conferencing, human-computer interaction*

## ⚙️ Step 1: Setup Environment

In [ ]:
# ── Check GPU ──────────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
!pip install -q lpips scikit-image gradio imageio imageio-ffmpeg einops
!git clone -q https://github.com/hzwer/Practical-RIFE
print('✅ Dependencies installed')

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

## 📥 Step 2: Download Pretrained Weights

In [ ]:
import os

# ── SAM2 Large Checkpoint ──────────────────────────────────────────────
os.makedirs('checkpoints', exist_ok=True)
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
print('✅ SAM2 weights downloaded')

# ── RIFE v4.25 Weights ─────────────────────────────────────────────────
# Download from Google Drive (Practical-RIFE model list):
# https://github.com/hzwer/Practical-RIFE#model-list
# Place the train_log/ folder inside Practical-RIFE/

# Example (if you have the file on Drive):
RIFE_DRIVE_PATH = '/content/drive/MyDrive/rife_v4.25_train_log.zip'  # ← Update this
if os.path.exists(RIFE_DRIVE_PATH):
    !unzip -q {RIFE_DRIVE_PATH} -d Practical-RIFE/
    print('✅ RIFE weights loaded from Drive')
else:
    print('⚠️ Please upload RIFE weights to Drive and update RIFE_DRIVE_PATH')

## 📁 Step 3: Clone TeleConf-Enhance

In [ ]:
!git clone -q https://github.com/your-username/teleconf-enhance
%cd teleconf-enhance
print('✅ Repo cloned')

## 🧪 Step 4: Run Pipeline on Sample Video

In [ ]:
from pipeline import run_pipeline

# Upload a sample video or use a file from Drive
INPUT_VIDEO  = 'assets/sample.mp4'  # ← Change to your video path
OUTPUT_VIDEO = 'assets/output.mp4'

run_pipeline(
    input_path       = INPUT_VIDEO,
    output_path      = OUTPUT_VIDEO,
    bg_removal       = True,
    interpolation    = True,
    scale            = 2,          # 30fps → 60fps
    background       = 'white',
    sam2_checkpoint  = '../checkpoints/sam2.1_hiera_large.pt',
    sam2_cfg         = 'sam2_hiera_l.yaml',
    rife_model_path  = '../Practical-RIFE/train_log',
)

In [ ]:
# ── Display side-by-side comparison ───────────────────────────────────
from IPython.display import HTML

HTML(f'''
<div style="display:flex; gap:20px; align-items:center">
  <div>
    <h3>Before</h3>
    <video width="400" controls>
      <source src="{INPUT_VIDEO}" type="video/mp4">
    </video>
  </div>
  <div>
    <h3>After (TeleConf-Enhanced)</h3>
    <video width="400" controls>
      <source src="{OUTPUT_VIDEO}" type="video/mp4">
    </video>
  </div>
</div>
''')

## 📊 Step 5: Evaluate Quality Metrics

In [ ]:
from evaluate import VideoEvaluator

# Compare interpolated output against ground truth (if available)
GT_VIDEO = 'assets/sample_gt_60fps.mp4'  # ← Ground truth 60fps video

if __import__('os').path.exists(GT_VIDEO):
    evaluator = VideoEvaluator()
    metrics = evaluator.evaluate_video(OUTPUT_VIDEO, GT_VIDEO)
    print(f'PSNR:  {metrics["psnr"]:.2f} dB')
    print(f'SSIM:  {metrics["ssim"]:.4f}')
    print(f'LPIPS: {metrics.get("lpips", "N/A")}')
else:
    print('Ground truth not found — skipping metric evaluation.')

## 🌐 Step 6: Launch Gradio Demo (Public URL)

In [ ]:
# Launches Gradio app with a public shareable link
!python app.py \
    --sam2_ckpt ../checkpoints/sam2.1_hiera_large.pt \
    --sam2_cfg  sam2_hiera_l.yaml \
    --rife_path ../Practical-RIFE/train_log \
    --share